### Preparing the tables 
#### 4 tables have been created - 
##### For **PYSPARK** - continent, country (created at test.pyspark.... in untiy catalog)
##### For **SQL** - continent, country (created at test.sql.... in untiy catalog)

In [0]:
%sql
create schema if not exists test.pyspark;
create schema if not exists test.sql;

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema_continent = StructType([
    StructField("continent", StringType(), True),
    StructField("population", IntegerType(), True)
])

data_continent = [("ASIA", 100000000), ("AFRICA", 999999999)]
df_continent = spark.createDataFrame(data_continent, schema=schema_continent)

df_continent.write.mode("overwrite").saveAsTable("test.pyspark.continent")

# Adding new rows to the table
data_continent_new = [("EUROPE", 74245280)]
df_continent_new = spark.createDataFrame(data_continent_new, schema=schema_continent)
df_continent_new.write.mode("append").saveAsTable("test.pyspark.continent")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema_country = StructType([
    StructField("country", StringType(), True),
    StructField("continent_name", StringType(), True),
    StructField("population", IntegerType(), True)
])

data_country = [("INDIA", 'ASIA', 123456789), ("NIGERIA", "AFRICA", 987654321)]
df_country = spark.createDataFrame(data_country, schema=schema_country)

df_country.write.mode("overwrite").saveAsTable("test.pyspark.country")

In [0]:
%sql
USE test.sql;
-- Define the continent table with proper columns
CREATE TABLE IF NOT EXISTS continent(
  continent VARCHAR(50),
  population INT
);

CREATE TABLE IF NOT EXISTS continent_incremental(
  continent VARCHAR(50),
  population INT
);

/* This is simple inserting logic which will add duplicate records into table */
INSERT INTO continent_incremental (continent, population)
VALUES 
  ('ASIA', 331002651), 
  ('AFRICA', 99999999),
  ('EUROPE', 74245280);

/* This is upsert logic to avoid duplicate entries in table */
MERGE INTO continent as target USING (SELECT DISTINCT continent, population FROM continent_incremental) as source
on source.continent = target.continent
WHEN MATCHED THEN
  UPDATE SET target.population = source.population
WHEN NOT MATCHED THEN
  INSERT (continent, population) 
  VALUES (source.continent, source.population);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,3,0,0


In [0]:
%sql
USE test.sql;
-- Define the continent table with proper columns
CREATE TABLE IF NOT EXISTS country(
  country VARCHAR(50),
  continent_name VARCHAR(50),
  population INT
);

CREATE TABLE IF NOT EXISTS country_incremental(
  country VARCHAR(50),
  continent_name VARCHAR(50),
  population INT
);

/* This is simple inserting logic which will add duplicate records into table */
INSERT INTO country_incremental (country, continent_name, population)
VALUES
  ('INDIA', 'ASIA', 331002651), 
  ('NIGERIA', 'AFRICA', 99999999);

/* This is upsert logic to avoid duplicate entries in table */
MERGE INTO country as target USING (SELECT DISTINCT country, continent_name, population FROM country_incremental) as source
on source.country = target.country
WHEN MATCHED THEN
  UPDATE SET target.population = source.population, target.continent_name = source.continent_name
WHEN NOT MATCHED THEN
  INSERT (country, continent_name, population) 
  VALUES (source.country, source.continent_name, source.population);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql
select * from country;

country,continent_name,population
INDIA,ASIA,331002651
NIGERIA,AFRICA,99999999


In [0]:
%sql
select * from continent;

continent,population
ASIA,331002651
AFRICA,99999999
EUROPE,74245280


###### INNER JOIN --> Used to find only common/matching records from both tables
###### LEFT OUTER JOIN --> Used to find all records from left table & only matching records from right table
###### RIGHT OUTER JOIN --> Used to find all records from right table & only matching records from left table
###### FULL OUTER JOIN --> Used to find all records from both tables
###### LEFT ANTI JOIN --> Used to find all records from left table excluding the matching ones with right table.
###### RIGHT ANTI JOIN --> Doesn't exists any join witgh this name in databricks SQL. We only have LEFT ANTI JOIN

In [0]:
%sql
select * from continent as c1 left join country as c2 on c1.continent = c2.continent_name;

continent,population,country,continent_name,population
ASIA,331002651,INDIA,ASIA,331002651
AFRICA,99999999,NIGERIA,AFRICA,99999999
EUROPE,74245280,null,null,null


In [0]:
%sql
select * from continent as c1 left join country as c2 on c1.continent = c2.continent_name and c1.continent = "ASIA";

continent,population,country,continent_name,population
ASIA,331002651,INDIA,ASIA,331002651
AFRICA,99999999,null,null,null
EUROPE,74245280,null,null,null


###### **NOTE:**
-- When we use only the ON condition in a LEFT JOIN,
-- all matching records are joined and all rows from the LEFT table are preserved.
-- Non-matching rows from the RIGHT table become NULL.

-- But when we add an extra condition using AND inside the ON clause,
-- the join matching becomes more restrictive.
-- Only rows satisfying BOTH conditions are matched.

-- Since this is still a LEFT JOIN,
-- all rows from the LEFT table will still appear,
-- but rows that do not satisfy the additional AND condition
-- will have NULL values for the RIGHT table columns.

In [0]:
%sql
select * from continent full outer join country on continent.continent = country.continent_name;

continent,population,country,continent_name,population
AFRICA,99999999,NIGERIA,AFRICA,99999999
ASIA,331002651,INDIA,ASIA,331002651
EUROPE,74245280,null,null,null


In [0]:
%sql
select * from country left anti join continent on country.continent_name = continent.continent;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5178011291666926>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select * from country left anti join continent on country.continent_name = continent.continent;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except Ba

In [0]:
df_continent = spark.table("test.pyspark.continent")
df_continent.display()
df_country.display()

continent,population
ASIA,100000000
AFRICA,999999999
EUROPE,74245280


country,continent_name,population
INDIA,ASIA,123456789
NIGERIA,AFRICA,987654321


In [0]:
from pyspark.sql.functions import col
df_continent.alias("c1").join(
    df_country.alias("c2"),
    col("c1.continent") == col("c2.continent_name"),
    'inner'
).filter(
    col("c2.country") != "INDIA"
).select(
    col("c1.continent"),
    col("c2.country"),
    col("c1.population")
).display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7492454794070659>, line 2
      1 from pyspark.sql.functions import col
----> 2 df_continent.alias("c1").join(
      3     df_country.alias("c2"),
      4     col("c1.continent") == col("c2.continent_name"),
      5     'inner'
      6 ).filter(
      7     col("c2.country") != "INDIA"
      8 ).select(
      9     col("c1.continent"),
     10     col("c2.country"),
     11     col("c1.population")
     12 ).display()

NameError: name 'df_continent' is not defined

In [0]:
from pyspark.sql.functions import col
df_continent.alias("c1").join(
    df_country.alias("c2"),
    col("c1.continent") == col("c2.continent_name"),
    'full_outer'
).select(
    col("c1.continent"),
    col("c2.country"),
    col("c1.population")
).display()

continent,country,population
AFRICA,NIGERIA,999999999
ASIA,INDIA,100000000
EUROPE,null,74245280


###### CARTESIAN PRODUCT --> Used to find all combinations of all records from both tables 
###### Its also called **CROSS JOIN** --> **(number of records/rows in table1) * (number of records/rows in table2)**
###### Here, there is of no need to mention "On" for the matching rows, sql automatically understands if mentioned 'cross join'.

In [0]:
%sql
select * from continent cross join country;

continent,population,country,continent_name,population
ASIA,331002651,INDIA,ASIA,331002651
AFRICA,99999999,NIGERIA,AFRICA,99999999
EUROPE,74245280,INDIA,ASIA,331002651
ASIA,331002651,NIGERIA,AFRICA,99999999
AFRICA,99999999,INDIA,ASIA,331002651
EUROPE,74245280,NIGERIA,AFRICA,99999999


In [0]:
from pyspark.sql.functions import col

df_continent.alias("c1").crossJoin(
    df_country.alias("c2")
).select("*"
    # col("c1.continent"),
    # col("c2.country"),
    # col("c1.population")
).display()

continent,population,country,continent_name,population
ASIA,100000000,INDIA,ASIA,123456789
AFRICA,999999999,INDIA,ASIA,123456789
EUROPE,74245280,INDIA,ASIA,123456789
ASIA,100000000,NIGERIA,AFRICA,987654321
AFRICA,999999999,NIGERIA,AFRICA,987654321
EUROPE,74245280,NIGERIA,AFRICA,987654321


In [0]:
# This is just a sample pyspark code to practise the full setup

from pyspark import col, when, concat, lit

df_match.alias("m") \
    .join(
        df_player.alias("p1"),
        col("m.winner_id") == col("p1.player_id"),
        'left'
    ) \
    .join(
        df_player.alias("p2"),
        col("m.runner_up_id") == col("p2.player_id"),
        'left'
    ) \
    .select(
        col("m.year"),
        col("m.tournamnet"),
        when(
            col("p1.player_id").isNull(),
            "Unknown"
        ) \
        .otherwise(
            concat(
                col("p1.first_name"),
                lit(" "),
                col("p1.last_name"),
                lit("("),
                col("p1.country"),
                lit(")")
            )
        ).alias("winner_name"),

        when(
            col("p2.player_id").isNull(),
            "Unknown"
        ) \
        .otherwise(
            concat(
                col("p2.first_name"),
                lit(" "),
                col("p2.last_name"),
                lit("("),
                col("p2.country"),
                lit(")"
            )
        ).alias("runner_up_name")
    ).display()